# PIBIC — métricas de embeddings e múltiplas referências (Colab)

Rode com **GPU**: *Ambiente de execução → Alterar tipo → T4 GPU*.

1. Prepara o repositório e os dados do FEB
2. Recalcula as métricas de n-gramas (rápido)
3. BERTScore + MoverScore
4. SBIC com múltiplas referências (OE2)
5. Correlações e download dos resultados

In [ ]:
!git clone -q https://github.com/caiobozzprocopilot/pibic.git
%cd pibic
!git clone -q --depth 1 https://github.com/allenai/feb.git data/raw/feb
!pip -q install -r requirements.txt -r requirements-colab.txt
import bert_score, rouge_score, sacrebleu; print('instalação ok')

In [ ]:
import nltk; nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)
!python src/carregar_feb.py
!python src/metricas_lexicais.py

## BERTScore e MoverScore
A célula abaixo funciona sozinha (clona o repositório se preciso), roda o BERTScore, as correlações e baixa o `resultados.zip`. O MoverScore fica numa célula separada e opcional.

In [ ]:
# 1) BERTScore — célula autossuficiente: prepara o ambiente se preciso, roda BERTScore e correlações
import os
if not os.path.exists('/content/pibic'):
    !git clone -q https://github.com/caiobozzprocopilot/pibic.git /content/pibic
%cd /content/pibic
if not os.path.exists('data/raw/feb'):
    !git clone -q --depth 1 https://github.com/allenai/feb.git data/raw/feb
!pip -q install -r requirements.txt -r requirements-colab.txt
import nltk; nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)
!python src/carregar_feb.py
!python src/metricas_lexicais.py
!python src/metricas_embeddings.py --so-bertscore
!python src/correlacoes.py
!zip -qr resultados.zip results
from google.colab import files; files.download('resultados.zip')

In [ ]:
# 2) MoverScore (opcional, precisa de GPU). Se der erro, pule esta célula.
!pip -q install pyemd moverscore
!python src/metricas_embeddings.py --so-moverscore

## Múltiplas referências (SBIC v2)

In [ ]:
!mkdir -p data/raw && cd data/raw && wget -q https://maartensap.com/social-bias-frames/SBIC.v2.tgz && tar xzf SBIC.v2.tgz && ls
# se os CSVs não ficarem em data/raw/SBIC.v2/, mova-os para lá
!mkdir -p data/raw/SBIC.v2 && mv -n data/raw/SBIC.v2.*.csv data/raw/SBIC.v2/ 2>/dev/null; ls data/raw/SBIC.v2
!python src/referencias_multiplas.py

## Correlações

In [ ]:
!python src/correlacoes.py

In [ ]:
import pandas as pd
pd.read_csv('results/correlacoes.csv').round(3)

## Baixar resultados
Depois, suba a pasta `results/` para o repositório (ou mande o zip para a conversa de Coordenação).

In [ ]:
!zip -qr resultados.zip results
from google.colab import files; files.download('resultados.zip')